<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/plot_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training Metrics Comparison

Load `0_metrics.json` from multiple training runs and compare selected metrics.

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ── Configure your runs here ──────────────────────────────────────────────────
# Each entry: (label, path to the training folder containing 0_metrics.json)

RUNS: list[tuple[str, str]] = [
    # ("run-1 (B=1k)", "/content/drive/MyDrive/models/exp_20260223_16-14/"),
    # ("run-2 (B=10k)", "/content/drive/MyDrive/models/exp_20260223_16-16/"),
    # ("run-3 (B=100k)", "/content/drive/MyDrive/models/exp_20260223_16-22/"),
    # ("run-4 (B=100k) #2", "/content/drive/MyDrive/models/exp_20260223_16-41/"),
    # ("run-5 (B=100)", "/content/drive/MyDrive/models/exp_20260223_16-58/"),
    # ("run-6 (B=1k) lr", "/content/drive/MyDrive/models/exp_20260223_23-44/"),
    # ("run-7 (B=10k) lr", "/content/drive/MyDrive/models/exp_20260223_23-48/"),
    # ("run-8 (B=10k) target", "/content/drive/MyDrive/models/exp_20260223_23-53/"),
    ("run-09 target baseline", "/content/drive/MyDrive/models/exp_20260224_12-26/"),
    ("run-10 target action", "/content/drive/MyDrive/models/exp_20260224_12-31/"),
    ("run-11 target γ=.99999", "/content/drive/MyDrive/models/exp_20260224_12-36/"),
]

# ── Global plot settings ──────────────────────────────────────────────────────
# X-axis mode: "step" for training steps, "time" for elapsed wallclock time (hours)
X_AXIS: str = "step"  # "step" or "time"

# Default axis limits (None = auto). Override per-plot via xlim/ylim kwargs.
XLIM: tuple[float, float] | None = None
YLIM: tuple[float, float] | None = None

# ── Load ──────────────────────────────────────────────────────────────────────
runs: dict[str, list[dict]] = {}
for label, folder in RUNS:
    p = Path(folder) / "0_metrics.json"
    if not p.exists():
        print(f"WARNING: {p} not found, skipping '{label}'")
        continue
    with p.open() as f:
        runs[label] = json.load(f)
    print(f"Loaded '{label}': {len(runs[label])} entries from {p}")

print(f"\n{len(runs)} run(s) loaded.")

## Training Parameters Comparison

Load `0_params.json` from each run and display a side-by-side comparison table.

In [ ]:
# ── Load and display training parameters ──────────────────────────────────────
import pandas as pd

# from techdays26.utils import extract_params_from_log

run_params: dict[str, dict] = {}
for label, folder in RUNS:
    p = Path(folder) / "0_params.json"
    if not p.exists():
        # Fallback: extract from 0_log.txt for older runs
        log_path = Path(folder) / "0_log.txt"
        if log_path.exists():
            print(f"'{label}': no 0_params.json, extracting from 0_log.txt...")
            extract_params_from_log(folder)
            p = Path(folder) / "0_params.json"
        else:
            print(
                f"WARNING: no 0_params.json or 0_log.txt in '{folder}', skipping '{label}'"
            )
            continue
    with p.open() as f:
        run_params[label] = json.load(f)

if run_params:
    df = pd.DataFrame(run_params).T  # rows = runs, columns = params
    df.index.name = "run"
    # Show only the most interesting columns (skip version strings etc.)
    display_cols = [
        c
        for c in df.columns
        if c
        not in (
            "commit_sha",
            "python_version",
            "pytorch_version",
            "techdays26_version",
            "ntuple_hash",
        )
    ]
    display(df[display_cols].T)
else:
    print("No 0_params.json files found.")

In [ ]:
# ── Helper: extract a metric as numpy arrays ─────────────────────────────────


def get_x_axis(data: list[dict]) -> tuple[np.ndarray, str]:
    """Return (x_values, x_label) based on the global X_AXIS setting.

    - "step": uses the 'step' field directly.
    - "time": computes cumulative wall time from 'wall_time_s' (in hours).
    """
    if X_AXIS == "time":
        wt = np.array([m.get("wall_time_s", 0.0) for m in data], dtype=np.float64)
        x = np.cumsum(wt) / 3600.0  # seconds → hours
        return x, "elapsed time (h)"
    else:
        x = np.array([m["step"] for m in data])
        return x, "step"


def get_metric(data: list[dict], key: str) -> tuple[np.ndarray, np.ndarray, str]:
    """Return (x, y, x_label) arrays for the given metric key."""
    x, x_label = get_x_axis(data)
    y = np.array([m[key] for m in data], dtype=np.float64)
    return x, y, x_label


def _apply_limits(ax, xlim, ylim):
    """Apply xlim/ylim to an axis, falling back to the global defaults."""
    xl = xlim if xlim is not None else XLIM
    yl = ylim if ylim is not None else YLIM
    if xl is not None:
        ax.set_xlim(*xl)
    if yl is not None:
        ax.set_ylim(*yl)


def plot_metric(
    metric_key: str,
    *,
    title: str | None = None,
    ylabel: str | None = None,
    xlabel: str | None = None,
    yscale: str = "linear",
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (10, 4),
    filter_inf: bool = True,
):
    """Plot a single metric across all loaded runs."""
    fig, ax = plt.subplots(figsize=figsize)
    x_label_auto = "step"
    for label, data in runs.items():
        x, y, x_label_auto = get_metric(data, metric_key)
        if filter_inf:
            mask = np.isfinite(y)
            x, y = x[mask], y[mask]
        ax.plot(x, y, marker=".", markersize=3, linewidth=1, label=label)
    ax.set_xlabel(xlabel or x_label_auto)
    ax.set_ylabel(ylabel or metric_key)
    ax.set_title(title or metric_key)
    ax.set_yscale(yscale)
    _apply_limits(ax, xlim, ylim)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_metrics_grid(
    metric_keys: list[str],
    *,
    ncols: int = 2,
    figsize_per_plot: tuple[float, float] = (6, 3.5),
    yscales: dict[str, str] | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    filter_inf: bool = True,
):
    """Plot multiple metrics in a grid of subplots."""
    yscales = yscales or {}
    n = len(metric_keys)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows),
        squeeze=False,
    )
    for idx, key in enumerate(metric_keys):
        ax = axes[idx // ncols][idx % ncols]
        x_label_auto = "step"
        for label, data in runs.items():
            x, y, x_label_auto = get_metric(data, key)
            if filter_inf:
                mask = np.isfinite(y)
                x, y = x[mask], y[mask]
            ax.plot(x, y, marker=".", markersize=2, linewidth=1, label=label)
        ax.set_title(key)
        ax.set_xlabel(x_label_auto)
        ax.set_yscale(yscales.get(key, "linear"))
        _apply_limits(ax, xlim, ylim)
        ax.legend(fontsize="small")
        ax.grid(True, alpha=0.3)
    # Hide unused subplots
    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)
    plt.tight_layout()
    plt.show()

## Loss

In [ ]:
plot_metric("loss", title="Training Loss (MSE)", ylabel="loss")

## Relative Weight Update ||dW|| / ||W||

In [ ]:
plot_metric(
    "rel_weight_update",
    title="Relative Weight Update  ||\u0394W|| / ||W||",
    ylabel="||\u0394W|| / ||W||",
    yscale="log",
)

## Value Function Statistics

In [ ]:
plot_metrics_grid(
    ["V_old_mean", "V_old_std", "V_old_abs_mean", "V_old_min", "V_old_max"],
    ncols=2,
)

## Weight and Gradient Norms

In [ ]:
plot_metrics_grid(
    ["W_norm", "delta_W_norm", "grad_mean", "grad_std"],
    ncols=2,
)

## Training Dynamics

In [ ]:
plot_metrics_grid(
    ["done_frac", "n_wins", "moves_left_mean", "lr"],
    ncols=2,
)

## Wall Time per Logging Interval

In [ ]:
plot_metric("wall_time_s", title="Wall Time per Interval", ylabel="seconds")

## Custom: Pick Any Metric

Available keys in each metrics entry:

In [ ]:
# Print all available metric keys
sample_run = next(iter(runs.values()))
print("Available metrics:")
for k in sample_run[0].keys():
    print(f"  - {k}")

In [ ]:
# Plot any metric by name:
# plot_metric("grad_nnz", title="Non-zero Gradient Entries")

---

# Arena Results Comparison

Load `0_arena_metrics.json` from each run and plot pre-computed scores over training steps.
Falls back to individual `step_X_arena_result.json` files for older runs.

In [ ]:
# ── Load arena results for each run from 0_arena_metrics.json ────────────────


def load_arena_results(folder: str) -> dict[int, list[dict]]:
    """Load arena aggregate scores from 0_arena_metrics.json.

    Falls back to reading individual step_X_arena_result.json files
    if the compact file doesn't exist (for older runs).

    Returns: {step: list_of_aggregate_rows}
    """
    folder = Path(folder)

    # Preferred: single compact file
    compact = folder / "0_arena_metrics.json"
    if compact.exists():
        with compact.open() as f:
            data = json.load(f)
        return {entry["step"]: entry["aggregates"] for entry in data}

    # Fallback: individual files (slow, for older runs)
    import re

    results = {}
    for p in sorted(folder.glob("step_*_arena_result.json")):
        m = re.search(r"step_(\d+)_arena_result", p.name)
        if m is None:
            continue
        step = int(m.group(1))
        with p.open() as f:
            data = json.load(f)
        results[step] = data["result"]["aggregates"]
    return dict(sorted(results.items()))


# Load arena results for all configured runs
arena_runs: dict[str, dict[int, list[dict]]] = {}
for label, folder in RUNS:
    ar = load_arena_results(folder)
    if not ar:
        print(f"WARNING: No arena result files found in '{folder}', skipping '{label}'")
        continue
    arena_runs[label] = ar
    steps = sorted(ar.keys())
    print(f"Loaded '{label}': {len(ar)} arena snapshots at steps {steps}")

print(f"\n{len(arena_runs)} run(s) with arena results.")

In [ ]:
# ── Arena helpers ─────────────────────────────────────────────────────────────


def get_matchup_key(row: dict) -> str:
    """Matchup identifier: 'agent_yellow vs agent_red'."""
    return f"{row['agent_yellow']} vs {row['agent_red']}"


def list_matchups(arena_data: dict[int, list[dict]]) -> list[str]:
    """List all unique matchup keys across all steps (epsilon=0 only)."""
    matchups = set()
    for rows in arena_data.values():
        for r in rows:
            if r["epsilon_yellow"] == 0.0 and r["epsilon_red"] == 0.0:
                matchups.add(get_matchup_key(r))
    return sorted(matchups)


def _step_to_time_map(label: str) -> dict[int, float]:
    """Build a mapping from step → cumulative wall time (hours) for a run."""
    if label not in runs:
        return {}
    data = runs[label]
    steps = np.array([m["step"] for m in data])
    wt = (
        np.cumsum(np.array([m.get("wall_time_s", 0.0) for m in data], dtype=np.float64))
        / 3600.0
    )
    return dict(zip(steps.tolist(), wt.tolist()))


def _map_steps_to_x(steps: list[int], label: str) -> tuple[list, str]:
    """Convert step list to x-values based on global X_AXIS setting."""
    if X_AXIS == "time":
        mapping = _step_to_time_map(label)
        x = [mapping.get(s, s) for s in steps]
        return x, "elapsed time (h)"
    return steps, "step"


def extract_series(
    arena_data: dict[int, list[dict]],
    matchup: str,
) -> dict[str, list]:
    """Extract time series for a specific matchup directly from the JSON data."""
    series: dict[str, list] = {"steps": []}

    for step, rows in sorted(arena_data.items()):
        for r in rows:
            if r["epsilon_yellow"] != 0.0 or r["epsilon_red"] != 0.0:
                continue
            if get_matchup_key(r) != matchup:
                continue
            series["steps"].append(step)
            for k in (
                "games",
                "yellow_wins",
                "red_wins",
                "draws",
                "score",
                "avg",
                "timeouts",
                "illegal_moves",
                "exceptions",
                "total_time_s",
            ):
                series.setdefault(k, []).append(r.get(k, 0))
            break  # one row per matchup per step

    return series


def plot_arena_single_run(
    run_label: str,
    metric: str = "avg",
    *,
    matchups: list[str] | None = None,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 5),
):
    """Plot a metric for a single run: one line per matchup."""
    if run_label not in arena_runs:
        print(f"Run '{run_label}' not found in arena_runs")
        return
    if matchups is None:
        matchups = list_matchups(arena_runs[run_label])

    fig, ax = plt.subplots(figsize=figsize)
    x_label = "step"
    for m in matchups:
        s = extract_series(arena_runs[run_label], m)
        if not s["steps"]:
            continue
        x, x_label = _map_steps_to_x(s["steps"], run_label)
        ax.plot(x, s[metric], marker="o", markersize=4, linewidth=1.5, label=m)

    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"[{run_label}] {metric}")
    _apply_limits(ax, xlim, ylim)
    ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
    ax.legend(fontsize="small", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_arena_compare_runs(
    matchup: str,
    metric: str = "avg",
    *,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 5),
):
    """Compare a metric for a specific matchup across runs: one line per run."""
    fig, ax = plt.subplots(figsize=figsize)

    x_label = "step"
    for label, arena_data in arena_runs.items():
        s = extract_series(arena_data, matchup)
        if not s["steps"]:
            continue
        x, x_label = _map_steps_to_x(s["steps"], label)
        ax.plot(x, s[metric], marker="o", markersize=4, linewidth=1.5, label=label)

    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"{matchup} — {metric}")
    _apply_limits(ax, xlim, ylim)
    ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_arena_compare_runs_grid(
    matchups: list[str],
    metric: str = "avg",
    *,
    title: str | None = None,
    ylabel: str | None = None,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
    figsize: tuple[float, float] = (12, 4),
):
    """One subplot per matchup, one line per run."""
    n = len(matchups)
    fig, axes = plt.subplots(n, 1, figsize=(figsize[0], figsize[1] * n), squeeze=False)

    x_label = "step"
    for idx, matchup in enumerate(matchups):
        ax = axes[idx][0]
        for label, arena_data in arena_runs.items():
            s = extract_series(arena_data, matchup)
            if not s["steps"]:
                continue
            x, x_label = _map_steps_to_x(s["steps"], label)
            ax.plot(
                x,
                s[metric],
                marker="o",
                markersize=4,
                linewidth=1.5,
                label=label,
            )
        ax.set_title(matchup)
        ax.set_xlabel(x_label)
        ax.set_ylabel(ylabel or metric)
        _apply_limits(ax, xlim, ylim)
        ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
        ax.legend(fontsize="small")
        ax.grid(True, alpha=0.3)

    plt.suptitle(title or f"{metric} per matchup", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
from collections import defaultdict

min_length = 10**10
first_label = list(arena_runs.keys())[-1]

starts_with = "ntuple"
ends_with = None

series_collection = defaultdict(list)
for k in arena_runs:
    for m in list_matchups(arena_runs[first_label]):
        if starts_with and not m.startswith(
            "ntuple"
        ):  # Filter for matchups where 'ntuple' is yellow
            continue
        if ends_with and not m.endswith(
            "ntuple"
        ):  # Filter for matchups where 'ntuple' is red
            continue
        series = extract_series(arena_runs[k], m)["avg"]
        series_collection[k].append(series)
        min_length = min(min_length, len(series))

# Average the series considering the minimum length
averaged_series: dict[str, np.ndarray] = {}
for label, list_of_series in series_collection.items():
    truncated_arrays = []
    for series in list_of_series:
        truncated_arrays.append(np.array(series[:min_length]))
    if truncated_arrays:
        averaged_series[label] = np.mean(np.array(truncated_arrays), axis=0)
    else:
        averaged_series[label] = np.array([])

print(f"Averaged series length: {min_length}")

In [ ]:
import matplotlib.pyplot as plt

# Get steps for x-axis (from one of the runs, truncated to min_length)
steps_for_plot = extract_series(
    arena_runs[first_label], list_matchups(arena_runs[first_label])[0]
)["steps"][:min_length]
x_for_plot, x_label_plot = _map_steps_to_x(steps_for_plot, first_label)

plt.figure(figsize=(12, 6))
for label, avg_vals in averaged_series.items():
    plt.plot(x_for_plot, avg_vals, marker="o", markersize=4, linewidth=1.5, label=label)

plt.xlabel(x_label_plot)
plt.ylabel("Averaged Score (ntuple vs others)")
plt.title("Averaged Arena Score Comparison (ntuple vs various opponents)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# List all available matchups (from last run)
first_label = list(arena_runs.keys())[-1]
print(f"Matchups in '{first_label}':")
for m in list_matchups(arena_runs[first_label]):
    print(f"  - {m}")

## Avg Score — All Matchups (Single Run)

In [ ]:
plot_arena_single_run(first_label, "avg")

## Compare Runs — Avg Score per Matchup

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "avg")
plot_arena_compare_runs("ntuple vs bitbully-16ply-book12ply", "avg")
plot_arena_compare_runs("bitbully-16ply-book12ply vs ntuple", "avg")

## Compare Runs — Avg Score Grid

In [ ]:
# All matchups in one grid (auto-detected from last run)
plot_arena_compare_runs_grid(
    list_matchups(arena_runs[first_label]), "avg", ylim=(-1.05, 1.05)
)

## Compare Runs — Yellow Wins

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "yellow_wins")
plot_arena_compare_runs("bitbully-full-strength vs ntuple", "yellow_wins")

## Compare Runs — Score

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "score")
plot_arena_compare_runs("bitbully-full-strength vs ntuple", "score")

## Custom Arena Plot

Use `plot_arena_compare_runs(matchup, metric)` with any matchup from the list above and any metric:
`avg`, `score`, `yellow_wins`, `red_wins`, `draws`, `games`, `timeouts`, `illegal_moves`, `exceptions`

In [ ]:
# plot_arena_compare_runs("ntuple vs random", "draws")